# 04 · Dicts and sets

After this notebook you can use dicts and sets for what they do best — constant-time lookup, counting, grouping and de-duplicating — explain *why* they are fast (hashing), and build a tiny inverted index: the data structure behind keyword search in RAG.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [03 · Lists and tuples](03_lists_and_tuples.ipynb)

## Why this matters in interviews

- "Use a dict or a set" is the answer to a large share of coding-round problems — two-sum, counting, grouping, de-duplication, caching — and it is what turns O(n²) into O(n).
- Interviewers probe the mechanism: hashing, why keys must be immutable, and what "O(1) on average" hides.
- In AI systems the same structures appear at scale: the inverted index behind BM25 keyword search, exact de-duplication of a corpus by content hash, caches keyed by `(model, prompt)`, tokenizer vocabularies.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve10` | What is an inverted index? |
| `dt09` | How do you deduplicate a large document corpus? (the exact-duplicate step) |

## What interviewers ask

- How does a dict work? Why is lookup O(1) on average, and what is the worst case?
- Why can't a list be a dict key when a tuple can? What makes an object hashable?
- Are dicts ordered?
- `d[k]` vs `d.get(k)` vs `setdefault` vs `defaultdict` — when do you use which?
- Count word frequencies and return the top k.
- Remove duplicates from a list while preserving order.
- Find the items common to two lists, or in one but not the other.
- Merge two dicts — which value wins for a duplicate key?

## 1 · Creating and reading a dict

A dict maps **keys** to **values**. Reading a missing key with `d[k]` raises `KeyError`; `d.get(k, default)` returns a default instead; `k in d` checks the **keys** (not the values).

In [ ]:
config = {"model": "gpt-4.1-mini", "temperature": 0.2, "max_tokens": 256}
also = dict(model="gpt-4.1-mini", temperature=0.2)        # keyword form: keys must be identifiers
pairs = dict(zip(["top_k", "alpha"], [5, 0.5]))            # from two parallel lists

print("config['model']         :", config["model"])
print("config.get('top_p')     :", config.get("top_p"))
print("config.get('top_p', 1.0):", config.get("top_p", 1.0))
print("'temperature' in config :", "temperature" in config, "| 0.2 in config:", 0.2 in config)
try:
    config["top_p"]
except KeyError as err:
    print("KeyError:", err)

config["top_p"] = 0.9                                       # insert (or overwrite)
removed = config.pop("max_tokens")                          # remove and return
missing = config.pop("seed", None)                          # no KeyError when you give a default
print("after insert / pop:", config, "| popped:", removed, missing)
assert "max_tokens" not in config and missing is None and pairs == {"top_k": 5, "alpha": 0.5}

`setdefault(k, default)` returns the value for `k`, inserting `default` first if `k` is missing — a one-line "get or create". `dict.fromkeys(keys, value)` maps every key to the **same** value object: fine for `0`, the `[[0] * 3] * 3` trap again for `[]`.

**Predict, then run:** what does `shared` contain after one append?

In [ ]:
by_source = {}
for chunk, source in [("c1", "faq"), ("c2", "policy"), ("c3", "faq")]:
    by_source.setdefault(source, []).append(chunk)
print("setdefault grouping:", by_source)

counts = dict.fromkeys(["user", "assistant"], 0)
shared = dict.fromkeys(["user", "assistant"], [])
shared["user"].append("hi")
print("fromkeys(..., 0) :", counts)
print("fromkeys(..., []):", shared)
assert by_source == {"faq": ["c1", "c3"], "policy": ["c2"]}
assert shared["assistant"] == ["hi"]                 # one list shared by every key

## 2 · Iterating: `keys()`, `values()`, `items()`

Iterating a dict yields its keys; `items()` yields `(key, value)` pairs — the usual way to loop. All three are **views**: live windows onto the dict, not copies. You may change *values* while iterating, but adding or removing keys raises `RuntimeError` — iterate over a snapshot (`list(d)`) when you need to delete.

**Predict, then run:** what does `keys_view` show after the insert?

In [ ]:
usage = {"prompt": 120, "completion": 45}
keys_view = usage.keys()
for name, tokens in usage.items():
    print(f"{name:<11}{tokens:>5}")
usage["cached"] = 30
print("keys_view after the insert:", keys_view)            # the view is live

try:
    for key in usage:
        if usage[key] < 50:
            del usage[key]
except RuntimeError as err:
    print("RuntimeError:", err)

for key in list(usage):                                     # iterate over a snapshot of the keys
    if usage[key] < 50:
        del usage[key]
print("after the safe delete:", usage)
assert list(keys_view) == ["prompt"] and usage == {"prompt": 120}

## 3 · Dicts remember insertion order

Since Python 3.7 the language guarantees that a dict iterates in **insertion order**. Updating an existing key keeps its position; deleting and re-inserting moves it to the end; `popitem()` removes the most recent pair. Equality ignores order. `collections.OrderedDict` adds order-sensitive equality and `move_to_end`, the core of an LRU cache ([LRU cache](../03_dsa/14_tries_union_find_lru_cache.ipynb)).

In [ ]:
from collections import OrderedDict

steps = {}
for step in ["retrieve", "rerank", "generate"]:
    steps[step] = "done"
steps["retrieve"] = "cached"                                 # update: position unchanged
print("order     :", list(steps))
print("popitem() :", steps.popitem())
print("{'a': 1, 'b': 2} == {'b': 2, 'a': 1}:", {"a": 1, "b": 2} == {"b": 2, "a": 1})
print("OrderedDict equality is order-sensitive:", OrderedDict(a=1, b=2) == OrderedDict(b=2, a=1))

lru = OrderedDict.fromkeys(["q1", "q2", "q3"])
lru.move_to_end("q1")                                        # q1 was just used
print("LRU order, oldest first:", list(lru))
assert list(steps) == ["retrieve", "rerank"] and list(lru) == ["q2", "q3", "q1"]

## 4 · Dict comprehensions and merging

A dict comprehension builds a dict from any iterable: `{key: value for item in items if condition}` (more in [07](07_comprehensions_iterators_generators.ipynb)).

To merge, the **right-hand side wins** on duplicate keys:

| Syntax | Result |
|---|---|
| `a \| b` (3.9+) | a new dict |
| `a \|= b` | updates `a` in place (like `a.update(b)`) |
| `{**a, **b}` | a new dict (older Pythons too, and can mix in literal keys) |

A classic use: request parameters = house defaults, overridden by whatever the caller passed. Merging is **shallow** — a nested dict on the right replaces the left one wholesale; it is not merged key by key.

In [ ]:
vocab = {tok: i for i, tok in enumerate(["<pad>", "the", "cat", "sat"])}
id_to_tok = {i: tok for tok, i in vocab.items()}                  # invert a one-to-one mapping
long_words = {w: len(w) for w in ["retrieval", "rag", "embedding"] if len(w) > 3}
print("vocab:", vocab, "| inverse:", id_to_tok, "| filtered:", long_words)

DEFAULTS = {"model": "gpt-4.1-mini", "temperature": 0.7, "extra": {"seed": 1, "user": "anon"}}
overrides = {"temperature": 0.0, "extra": {"seed": 42}}
params = DEFAULTS | overrides
print("DEFAULTS | overrides:", params)
print("same as {**a, **b}  :", {**DEFAULTS, **overrides} == params)
assert params["temperature"] == 0.0 and params["extra"] == {"seed": 42}     # nested dict replaced, not merged
assert DEFAULTS["temperature"] == 0.7                                       # `|` did not touch DEFAULTS

## 5 · `defaultdict` and `Counter`

- `collections.defaultdict(factory)` calls `factory()` to create a value the first time a missing key is **read** — `defaultdict(list)` for grouping, `defaultdict(int)` for counting. Trap: merely *looking up* a missing key inserts it. (`.get()` does not.)
- `collections.Counter` is a dict subclass for counting hashable things: `most_common(k)`, arithmetic between counters, `total()` (3.10+). A missing key reads as `0` **without** being inserted.

**Predict, then run:** what is `len(groups)` after reading `groups["blog"]`?

In [ ]:
from collections import Counter, defaultdict

groups = defaultdict(list)
for chunk_id, doc in [("c1", "faq"), ("c2", "policy"), ("c3", "faq")]:
    groups[doc].append(chunk_id)
print("grouped:", dict(groups), "| len:", len(groups))
print("groups.get('blog')  :", groups.get("blog"), "| len:", len(groups))
print("groups['blog']      :", groups["blog"], "| len:", len(groups))

roles = Counter(["user", "assistant", "user", "tool", "user"])
print("Counter:", roles, "| most_common(1):", roles.most_common(1), "| roles['system']:", roles["system"])
print("total():", roles.total(), "| roles + roles:", roles + roles)
assert len(groups) == 3 and "system" not in roles and roles.most_common(1) == [("user", 3)]

In [ ]:
import re

import matplotlib.pyplot as plt

documents = {
    0: "Refunds are processed within five business days of the return.",
    1: "To request a refund, contact support with your order number.",
    2: "Support is available on business days by chat and email.",
    3: "Refunds over 500 dollars need manager approval before the refund is issued.",
    4: "Your order number is in the confirmation email for every order.",
}
STOPWORDS = {"a", "and", "are", "before", "by", "for", "in", "is", "of", "on", "over", "the", "to", "with", "your"}

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

freq = Counter(tok for text in documents.values() for tok in tokenize(text) if tok not in STOPWORDS)
top = freq.most_common(8)
print(top)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.barh([w for w, _ in top][::-1], [c for _, c in top][::-1], color="#4c6ef5")
ax.set_title("Top words in a five-document support corpus (Counter.most_common)")
ax.set_xlabel("count"); ax.set_xticks(range(0, top[0][1] + 1))
plt.show()
assert top[0] == ("order", 3) and freq["refund"] == 2 and freq["refunds"] == 2

Notice `refund` and `refunds` are counted separately: a plain tokenizer does no *stemming*. Search engines run an analyzer (lowercase, stem, drop stopwords) before indexing for exactly this reason.

## 6 · Why dicts and sets are fast: hashing

**In plain English:** a dict is an array of slots. To store a key, Python computes `hash(key)` — a large integer — and uses it to jump straight to a slot, instead of scanning. Two keys that land on the same slot (a *collision*) are resolved by probing further slots, and when the table is about two-thirds full it is resized. Average cost of a lookup: **O(1)**. Worst case, if many keys collide: O(n) — rare with good hash functions.

```
key          hash(key) % 8     slot            (slot numbers are illustrative)
"refund"  ─────────────────►    3   ──►  ("refund", ...)
"policy"  ─────────────────►    6   ──►  ("policy", ...)
"faq"     ─────────────────►    3   ──►  collision: probe another slot
```

What follows from this:

- A key must be **hashable**, and its hash must never change. That is why mutable containers (`list`, `dict`, `set`) cannot be keys, while `str`, numbers and tuples of hashables can.
- Keys that are **equal must hash equal**, so `1`, `1.0` and `True` are the same key.
- String hashes are **randomised per process** (to defeat collision attacks), so never store `hash(s)` — use `hashlib` for stable fingerprints.

In [ ]:
import hashlib
import os
import subprocess
import sys

print("hash(42) == 42:", hash(42) == 42, "| hash(1) == hash(1.0) == hash(True):", hash(1) == hash(1.0) == hash(True))
for bad in ([1, 2], {"a": 1}, (1, [2])):
    try:
        hash(bad)
    except TypeError as err:
        print(f"hash({bad!r}) -> TypeError: {err}")

cache = {("gpt-4.1-mini", "What is RAG?", 0.0): "RAG = retrieval + generation"}   # tuple key: fine
print("{1: 'a', 1.0: 'b', True: 'c'} ->", {1: "a", 1.0: "b", True: "c"})

env = {k: v for k, v in os.environ.items() if k != "PYTHONHASHSEED"}
run = lambda code: subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, env=env).stdout.strip()
builtin_hashes = {run("print(hash('refund'))") for _ in range(2)}
sha_hashes = {run("import hashlib; print(hashlib.sha256(b'refund').hexdigest()[:16])") for _ in range(2)}
print("hash('refund') in two fresh processes :", len(builtin_hashes), "different values")
print("sha256('refund') in two fresh processes:", len(sha_hashes), "value ->", next(iter(sha_hashes)))
assert len(builtin_hashes) == 2 and len(sha_hashes) == 1

## 7 · Sets and set algebra

A set is an unordered collection of **unique**, hashable items — a dict with keys only. Create one with `{1, 2}` or `set(iterable)`; note that `{}` is an empty **dict**. The algebra is where sets shine:

| Operation | Operator | Method |
|---|---|---|
| union (in either) | `a \| b` | `a.union(b)` |
| intersection (in both) | `a & b` | `a.intersection(b)` |
| difference (in `a`, not `b`) | `a - b` | `a.difference(b)` |
| symmetric difference (in exactly one) | `a ^ b` | `a.symmetric_difference(b)` |
| subset / superset | `a <= b`, `a >= b` | `issubset`, `issuperset` |

The methods accept any iterable (`a.union(["x"])`); the operators need sets on both sides. Sets of strings iterate in an order that can change between runs (hash randomisation again), so **sort before printing** or comparing as lists.

In [ ]:
query = {"refund", "policy", "days", "late"}
doc = {"refund", "processed", "business", "days", "policy"}
print("type({}):", type({}).__name__, "| type(set()):", type(set()).__name__)
print("in both    :", sorted(query & doc))
print("in either  :", sorted(query | doc))
print("query only :", sorted(query - doc))
print("exactly one:", sorted(query ^ doc))
print("subset?", {"refund", "days"} <= doc, "| disjoint?", query.isdisjoint({"shipping"}))
jaccard = len(query & doc) / len(query | doc)
print(f"Jaccard similarity |A ∩ B| / |A ∪ B| = {jaccard:.2f}")

tags = {"rag"}
tags.add("rag"); tags.add("bm25"); tags.discard("missing")        # add is idempotent; discard never raises
print("tags:", sorted(tags))
assert query & doc == {"refund", "policy", "days"} and query - doc == {"late"} and len(tags) == 2

In [ ]:
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.add_patch(Circle((0.38, 0.5), 0.3, alpha=0.35, color="#4c6ef5"))
ax.add_patch(Circle((0.62, 0.5), 0.3, alpha=0.35, color="#f59f00"))
ax.text(0.22, 0.5, "\n".join(sorted(query - doc)), ha="center", va="center", fontsize=10)
ax.text(0.50, 0.5, "\n".join(sorted(query & doc)), ha="center", va="center", fontsize=10)
ax.text(0.80, 0.5, "\n".join(sorted(doc - query)), ha="center", va="center", fontsize=10)
ax.text(0.26, 0.86, "query terms", ha="center"); ax.text(0.74, 0.86, "document terms", ha="center")
ax.set_xlim(0, 1); ax.set_ylim(0.12, 0.92); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"query & doc is the overlap in the middle; Jaccard = {jaccard:.2f}")
plt.show()

## 8 · Membership: set O(1) vs list O(n)

`x in some_list` compares against every element until it finds a match — O(n), and the worst case (the item is absent) scans everything. `x in some_set` hashes `x` and checks one slot — O(1) on average. Building the set costs O(n) once, so converting pays off as soon as you do more than a handful of lookups.

In [ ]:
import timeit

ns = [10, 100, 1_000, 10_000, 100_000]
list_us, set_us = [], []
for n in ns:
    data_list = list(range(n))
    data_set = set(data_list)
    reps = max(5, 200_000 // n)
    list_us.append(min(timeit.repeat(lambda: -1 in data_list, number=reps, repeat=3)) / reps * 1e6)
    set_us.append(min(timeit.repeat(lambda: -1 in data_set, number=5_000, repeat=3)) / 5_000 * 1e6)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.loglog(ns, list_us, marker="o", label="x in list")
ax.loglog(ns, set_us, marker="s", label="x in set")
ax.set_title("Looking up an absent item: list time grows with n, set time stays flat")
ax.set_xlabel("collection size n"); ax.set_ylabel("time per lookup (µs)"); ax.legend()
plt.show()
print(f"n = {ns[-1]:,}: list {list_us[-1]:.0f} µs vs set {set_us[-1]:.3f} µs per lookup")
assert list_us[-1] > 50 * set_us[-1]

## 9 · `frozenset`: a hashable set

`frozenset` is an immutable set, so it can be a dict key or an element of another set — the natural key for an **unordered pair**, such as "these two documents are near-duplicates" (the pair `{a, b}` is the same as `{b, a}`).

In [ ]:
dupe_pairs = {frozenset({"doc-3", "doc-9"}), frozenset({"doc-9", "doc-3"}), frozenset({"doc-1", "doc-4"})}
print("unique pairs:", len(dupe_pairs))
similarity = {frozenset({"doc-3", "doc-9"}): 0.97}
print("look up in either order:", similarity[frozenset({"doc-9", "doc-3"})])
try:
    {{"a"}}                                  # a set cannot contain a (mutable) set
except TypeError as err:
    print("TypeError:", err)
assert len(dupe_pairs) == 2

## 10 · Practical patterns

### 10.1 Order-preserving de-duplication (and exact dedupe of a corpus)

`list(set(items))` de-duplicates but loses the order. `list(dict.fromkeys(items))` keeps the first occurrence of each item, in order — dict keys are unique *and* ordered. When "duplicate" needs a rule — ignore case, compare normalised text — use a `seen` set with a key function.

For a **corpus** (`dt09`) the exact-duplicate step is the same idea at scale: normalise each document, fingerprint it with `hashlib` (stable across runs, unlike `hash()`), and keep a set of fingerprints — O(n) time, storing 32-byte digests instead of whole documents. Near-duplicates need MinHash / LSH, which estimates the Jaccard similarity from section 7 without comparing every pair.

In [ ]:
queries = ["reset password", "Reset Password", "refund status", "reset password", "shipping"]
print("dict.fromkeys  :", list(dict.fromkeys(queries)))

def dedupe(items, key=lambda x: x):
    seen, out = set(), []
    for item in items:
        k = key(item)
        if k not in seen:
            seen.add(k)
            out.append(item)
    return out

print("dedupe casefold:", dedupe(queries, key=str.casefold))

def fingerprint(text: str) -> str:
    normalised = " ".join(text.casefold().split())
    return hashlib.sha256(normalised.encode("utf-8")).hexdigest()

corpus = ["Refunds take 5 days.", "refunds  take 5 DAYS.", "Shipping is free.", "Refunds take 5 days."]
unique_docs = dedupe(corpus, key=fingerprint)
print("unique docs    :", unique_docs, f"({len(corpus) - len(unique_docs)} duplicates dropped)")
assert dedupe(queries, key=str.casefold) == ["reset password", "refund status", "shipping"]
assert unique_docs == ["Refunds take 5 days.", "Shipping is free."]

### 10.2 A tiny inverted index — keyword search, RAG preview (`ve10`)

An **inverted index** maps each term to the set of documents that contain it, like the index at the back of a book. A query becomes set algebra on those *posting lists*: AND is an intersection, OR is a union. Ranking the survivors by how many query terms they contain is a crude version of what BM25 does properly ([BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb)). Note the `.get(term, ...)` when querying a `defaultdict`: plain `index[term]` would insert every unknown query term.

In [ ]:
index = defaultdict(set)
for doc_id, text in documents.items():
    for term in tokenize(text):
        index[term].add(doc_id)

print("posting list 'support':", sorted(index["support"]))
print("posting list 'order'  :", sorted(index["order"]))
print("vocabulary size       :", len(index))

def search_and(query):
    postings = [index.get(t, set()) for t in tokenize(query)]
    return sorted(set.intersection(*postings)) if postings else []

def search_or(query):
    return sorted(set().union(*(index.get(t, set()) for t in tokenize(query))))

print("AND 'order number'    :", search_and("order number"))
print("OR  'refund shipping' :", search_or("refund shipping"))
print("AND 'refund approval' :", search_and("refund approval"))
assert search_and("order number") == [1, 4] and search_or("refund shipping") == [1, 3] and search_and("refund approval") == [3]

In [ ]:
def rank(query, k=3):
    hits = Counter()
    for term in dict.fromkeys(tokenize(query)):          # unique terms, in a fixed order
        for doc_id in index.get(term, ()):
            hits[doc_id] += 1
    return hits.most_common(k)

for q in ["business days support", "refund order", "refunds"]:
    print(f"{q!r:<24} -> {rank(q)}")
assert rank("business days support")[0] == (2, 3)
assert "shipping" not in index and len(index) == len({t for text in documents.values() for t in tokenize(text)})

### 10.3 Reading nested JSON safely

LLM and API responses are nested dicts and lists. `resp["choices"][0]["message"]["content"]` is fine when you *know* the shape. When fields may be missing, chain `.get()` with empty defaults, or use a small helper that walks a path and returns a default on any miss.

One trap deserves its own line: `d.get("tool_calls", [])` returns `None` — not `[]` — when the key **exists** with the value `None`, which is exactly how chat APIs report "no tool calls". Then `for call in ...` crashes. Use `d.get("tool_calls") or []`.

In [ ]:
response = {"choices": [{"message": {"role": "assistant", "content": "Hi!", "tool_calls": None}}],
            "usage": {"prompt_tokens": 12}}

message = response["choices"][0]["message"]
print("direct             :", message["content"])
print("chained .get       :", response.get("usage", {}).get("completion_tokens", 0))
print("get(key, [])       :", message.get("tool_calls", []), " <- the key exists, its value is None")
print("get(key) or []     :", message.get("tool_calls") or [])

def dig(data, *path, default=None):
    """Follow keys / indexes; return `default` if any step is missing."""
    for step in path:
        try:
            data = data[step]
        except (KeyError, IndexError, TypeError):
            return default
    return data

print("dig content        :", dig(response, "choices", 0, "message", "content"))
print("dig missing index  :", dig(response, "choices", 3, "message", default="<no choice 3>"))
print("dig through None   :", dig(response, "choices", 0, "message", "tool_calls", 0, default="<none>"))
assert dig(response, "usage", "prompt_tokens") == 12 and dig(response, "x", "y") is None
assert message.get("tool_calls", []) is None and (message.get("tool_calls") or []) == []

## Try it yourself

**1.** Group these words by their first letter, with each group sorted: `["rag", "rerank", "bm25", "retriever", "batch", "embedding"]`.

In [ ]:
# Solution — try it yourself first, then run this
words = ["rag", "rerank", "bm25", "retriever", "batch", "embedding"]
by_letter = defaultdict(list)
for w in words:
    by_letter[w[0]].append(w)
result = {letter: sorted(ws) for letter, ws in sorted(by_letter.items())}
print(result)
assert result == {"b": ["batch", "bm25"], "e": ["embedding"], "r": ["rag", "rerank", "retriever"]}

**2.** Two-sum, AI-flavoured: given the token counts of several chunks and a budget, return the indices of two chunks that exactly fill the budget — in **one pass**, O(n), with a dict.

In [ ]:
# Solution — try it yourself first, then run this
def two_sum(counts, target):
    seen = {}                                   # token count -> index where we saw it
    for i, c in enumerate(counts):
        if target - c in seen:
            return seen[target - c], i
        seen[c] = i
    return None

counts = [300, 812, 450, 700, 212]
print(two_sum(counts, 1_000))
assert two_sum(counts, 1_000) == (0, 3) and two_sum(counts, 5) is None

**3.** Write `jaccard(text_a, text_b)` over lower-cased word sets. Identical texts should score 1.0, texts with no shared words 0.0 — and two empty texts should not crash.

In [ ]:
# Solution — try it yourself first, then run this
def jaccard(text_a: str, text_b: str) -> float:
    a, b = set(tokenize(text_a)), set(tokenize(text_b))
    return len(a & b) / len(a | b) if a | b else 1.0

print(round(jaccard("Refunds take five days", "refunds take 5 business days"), 3))
assert jaccard("same words", "Same words") == 1.0 and jaccard("cat", "dog") == 0.0 and jaccard("", "") == 1.0

**4.** Add a NOT to the inverted index: `search_and_not("business days", "support")` returns the documents that contain every term of the first query but none of the second.

In [ ]:
# Solution — try it yourself first, then run this
def search_and_not(must: str, must_not: str) -> list[int]:
    excluded = set().union(*(index.get(t, set()) for t in tokenize(must_not)))
    return sorted(set(search_and(must)) - excluded)

print(search_and_not("business days", "support"))
assert search_and_not("business days", "support") == [0] and search_and_not("order", "email") == [1]

## Say it in an interview

- **How a dict works (30 s):** "A dict is a hash table: `hash(key)` picks a slot, collisions are resolved by probing, and the table resizes at about two-thirds full, so lookup, insert and delete are O(1) on average and O(n) in the pathological worst case. Since 3.7 dicts preserve insertion order. Keys must be hashable — equal keys must hash equal and the hash must never change — which is why lists can't be keys but tuples of immutables can."
- **Which tool:** `d[k]` when missing is a bug; `get(k, default)` when missing is normal; `setdefault` or `defaultdict(list)` to group; `Counter` to count and for top-k; `set` for membership and dedupe; `frozenset` for unordered pairs or set-valued keys.
- **Dedupe:** `list(dict.fromkeys(xs))` keeps order; a `seen` set with a key function handles "equal after normalising"; at corpus scale, SHA-256 of normalised text for exact duplicates, then MinHash/LSH for near-duplicates.
- **Inverted index:** term → posting list of document ids; AND = intersect, OR = union; that is the core of BM25 / Elasticsearch, and why exact identifiers are cheap to find.
- **Traps:** `defaultdict` inserts on read; `dict.fromkeys(keys, [])` shares one list; `.get(k, [])` returns `None` when the value is `None`; mutating a dict while iterating it; `1`, `1.0`, `True` collide; `{}` is a dict; `hash(str)` differs between runs; `|` merges are shallow.

## Next

- [05 · Control flow and loops](05_control_flow_and_loops.ipynb).
- [Hashing and sliding window](../03_dsa/03_hashing_and_sliding_window.ipynb) — dict/set patterns for coding rounds.
- [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) — the inverted index, scored properly.
- [Tries, union-find and LRU cache](../03_dsa/14_tries_union_find_lru_cache.ipynb) — `OrderedDict` put to work.
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 4: collections) · the [interview bank](../../ai_interview_prep/index.html) for `ve10` and `dt09`.